0. Imports.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, VARCHAR, CHAR, TEXT, DateTime, Integer, Float

1. Carregando os dados.

In [2]:
tables = {
    'customers': 'olist_order_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

olist_df = {}
for name, archive in tables.items():
    df = pd.read_csv(f'../data/raw/{archive}')
    olist_df[name] = df

2. olist_orders_dataset.csv.

- 2.2. Criando colunas (considerando que não houve conversão de tipos de dados ainda):
  *  delivery_delay_days: dias de atraso (int) na entrega, essencial para as perguntas de negócio 1, 2, 3 e 12;
  *  purchase_year_month: ano e mês da compra (str), facilita agregações temporais no SQL para as perguntas 1, 6 e 11;
  *  delivery_time_days: Dias entre a aprovação do pedido e a data estimada de entrega, útil para a pergunta de negócio 4;
  *  approval_to_carrier_days: Dias entre a aprovação do pedido e a entrega à transportadora, útil para a pergunta de negócio 5;
  *  full_delivery_days: Dias entre a compra e a entrega ao cliente, útil para a pergunta de negócio 14;
  *  estimated_vs_carrier_days: Dias entre a data estimada de entrega e a coleta pela transportadora, útil para a pergunta de negócio 16.

In [3]:
#Criando as colunas
olist_df['orders']['delivery_delay_days'] = (pd.to_datetime(olist_df['orders']['order_delivered_customer_date']) - pd.to_datetime(olist_df['orders']['order_estimated_delivery_date'])).dt.days
olist_df['orders']['purchase_year_month'] = pd.to_datetime(olist_df['orders']['order_purchase_timestamp']).dt.to_period('M').astype(str)
olist_df['orders']['delivery_time_days'] = (pd.to_datetime(olist_df['orders']['order_estimated_delivery_date']) - pd.to_datetime(olist_df['orders']['order_approved_at'])).dt.days
olist_df['orders']['approval_to_carrier_days'] = (pd.to_datetime(olist_df['orders']['order_delivered_carrier_date']) - pd.to_datetime(olist_df['orders']['order_approved_at'])).dt.days
olist_df['orders']['full_delivery_days'] = (pd.to_datetime(olist_df['orders']['order_delivered_customer_date']) - pd.to_datetime(olist_df['orders']['order_purchase_timestamp'])).dt.days
olist_df['orders']['estimated_vs_carrier_days'] = (pd.to_datetime(olist_df['orders']['order_estimated_delivery_date']) - pd.to_datetime(olist_df['orders']['order_delivered_carrier_date'])).dt.days

- 2.2. Tipos de dados:
  *  Convertendo as colunas a partir de order_purchase_timestamp de str e para datetime;
  *  Mapeando as colunas.

In [5]:
#str -> datetime
columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
                    'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in columns:
    olist_df['orders'][col] = pd.to_datetime(olist_df['orders'][col])

#MAPEAMENTO SQL
dtype_orders = {
    'order_id': VARCHAR(32),
    'customer_id': VARCHAR(32),
    'order_status': VARCHAR(20),
    'order_purchase_timestamp': DateTime(),
    'order_approved_at': DateTime(),
    'order_delivered_carrier_date': DateTime(),
    'order_delivered_customer_date': DateTime(),
    'order_estimated_delivery_date': DateTime(),
    'delivery_delay_days': Integer(),
    'order_processing_time_days': Integer(),
    'purchase_year_month': VARCHAR(7),
    'delivery_time_days': Integer(),
    'approval_to_carrier_days': Integer(),
    'full_delivery_days': Integer(),
    'estimated_vs_carrier_days': Integer()
}

- 2.3. Valores nulos: Os nulos nas colunas 'order_approved_at', 'order_delivered_carrier_date' e 'order_delivered_customer_date' têm significado lógico (pedidos cancelados, não aprovados ou não entregues), portanto, os valores devem ser mantidos;
- 2.4. Nomes de colunas: Já estão padronizados;
- 2.5. Colunas duplicadas: Não há.

3. olist_products_dataset.

- 3.1. Valores nulos:
  *  product_category_name: remover linhas nulas, pois é foco das perguntas de negócio 3, 7 e 13;
  *  product_name_lenght, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm: imputar pela mediana, pois os nulos são numéricos sem significado lógico claro.

In [6]:
#Removendo linhas nulas em product_category_name (foco das perguntas de negócio)
olist_df['products'] = olist_df['products'].dropna(subset=['product_category_name'])

#Imputando pela mediana nas colunas numéricas restantes
numeric_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in numeric_cols:
    median = olist_df['products'][col].median()
    olist_df['products'][col] = olist_df['products'][col].fillna(median)

- 3.2. Tipos de dados: Mapeando as colunas.

In [7]:
#MAPEAMENTO SQL
dtype_products = {
    'product_id': VARCHAR(32),
    'product_category_name': VARCHAR(100),
    'product_name_lenght': Float(),
    'product_description_lenght': Float(),
    'product_photos_qty': Float(),
    'product_weight_g': Float(),
    'product_length_cm': Float(),
    'product_height_cm': Float(),
    'product_width_cm': Float()
}

- 3.3. Criar colunas: Sem necessidade;
- 3.4. Nomes de colunas: Já estão padronizados;
- 3.5 Colunas duplicadas: Não há.

Obs.: Há registros de ‘product_category_name’ que não existem na tabela ‘category_translations’. MySQL não permite criar a chave estrangeira (FK) em ‘products’ se houver um único ‘produto fantasma’ na tabela de produtos, portanto, foi necessário remover tais linhas.

In [8]:
#Filtra a tabela products para manter apenas categorias que existem na tradução
olist_df['products'] = olist_df['products'][
    olist_df['products']['product_category_name'].isin(olist_df['category_translation']['product_category_name'])
]

4. olist_sellers_dataset.

- 4.1. Tipos de dados: Mapeando as colunas.

In [9]:
#MAPEAMENTO SQL
dtype_sellers = {
    'seller_id': VARCHAR(32),
    'seller_zip_code_prefix': Integer(),
    'seller_city': VARCHAR(100),
    'seller_state': CHAR(2)
}

- 4.2. Valores nulos: Não há;
- 4.3. Criar colunas: Sem necessidade;
- 4.4. Nomes de colunas: Já estão padronizados;
- 4.5. Colunas duplicadas: Não há.

Obs.: seller_city possui inconsistências (como: 'lages - sc' e 'rio de janeiro, rio de janeiro, brasil'), portanto é necessário remover caracteres especiais, espaços extras e padronizar tudo em minúsculo para garantir consistência.

In [10]:
olist_df['sellers']['seller_city'] = (
    olist_df['sellers']['seller_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

5. product_category_name_translation.

- 5.1. Tipos de dados: Mapeando as colunas.

In [11]:
#MAPEAMENTO SQL
dtype_category_translation = {
    'product_category_name': VARCHAR(100),
    'product_category_name_english': VARCHAR(100)
}

- 5.2. Valores nulos: Não há;
- 5.3. Criar colunas: Sem necessidade;
- 5.4. Nomes de colunas: Já estão padronizados;
- 5.5. Colunas duplicadas: Não há.

6. olist_order_items_dataset.

- 6.1. Criar colunas: 
  *  total_item_value: Valor total pago por item incluindo frete, útil para as perguntas de negócio 15.

In [12]:
olist_df['order_items']['total_item_value'] = olist_df['order_items']['price'] + olist_df['order_items']['freight_value']

- 6.2. Tipos de dados: 
  *  shipping_limit_date: Convertendo de str para datetime;
  *  Mapeando as colunas.

In [13]:
#str -> datetime
olist_df['order_items']['shipping_limit_date'] = pd.to_datetime(olist_df['order_items']['shipping_limit_date'])

#MAPEAMENTO SQL
dtype_order_items = {
    'order_id': VARCHAR(32),
    'order_item_id': Integer(),
    'product_id': VARCHAR(32),
    'seller_id': VARCHAR(32),
    'shipping_limit_date': DateTime(),
    'price': Float(),
    'freight_value': Float(),
    'total_item_value': Float()
}

- 6.3. Valores nulos: Não há;
- 6.4. Nomes de colunas: Já estão padronizados;
- 6.5. Colunas duplicadas: Não há.

Obs.: Há registros de product_id que não existem na tabela products. MySQL não permitirá criar a chave estrangeira (FK) se houver um único "produto fantasma" na tabela de itens de pedido, portanto, tais linhas devem ser removidas.

In [14]:
#Filtrando 'order_items' para manter apenas produtos que existem na tabela products
#Limpa os "produtos órfãos"
olist_df['order_items'] = olist_df['order_items'][
    olist_df['order_items']['product_id'].isin(olist_df['products']['product_id'])
]

7. olist_order_payments_dataset.

- 7.1. Criar colunas: 
  *  payment_method: Classificação do pagamento como à vista ou parcelado, útil para a pergunta de negócio 10.


In [15]:
#Criando coluna
olist_df['order_payments']['payment_method'] = olist_df['order_payments']['payment_installments'].apply(lambda x: 'À Vista (1x)' if x == 1 else 'Parcelado (+ 1x)')

- 7.2. Tipos de dados: Mapeando as colunas.

In [16]:
#MAPEAMENTO SQL
dtype_order_payments = {
    'order_id': VARCHAR(32),
    'payment_sequential': Integer(),
    'payment_type': VARCHAR(20),
    'payment_installments': Integer(),
    'payment_value': Float(),
    'payment_method': VARCHAR(20)
}

- 7.3. Valores nulos: não há;
- 7.4. Nomes de colunas: Já estão padronizados;
- 7.5. Colunas duplicadas: Não há.

8. olist_order_reviews_dataset.

- 8.1. Valores nulos:
  *  review_comment_title: Nenhuma das 16 perguntas de negócio foca no conteúdo textual dos títulos. Portanto, preencher com 'Sem título';
  *  review_comment_message: Nenhuma das 16 perguntas de negócio foca no conteúdo textual das mensagens. Portanto, preencher com 'Sem mensagem'.

In [17]:
#Imputandos as linhas com valores nulos
olist_df['order_reviews']['review_comment_title'] = olist_df['order_reviews']['review_comment_title'].fillna('Sem título')
olist_df['order_reviews']['review_comment_message'] = olist_df['order_reviews']['review_comment_message'].fillna('Sem mensagem')

- 8.2. Tipos de dados:
  *  review_creation_date: Convertendo de str para datetime;
  *  review_answer_timestamp: Convertendo de str para datetime;
  *  Mapeando as colunas.

In [18]:
#str -> datetime
olist_df['order_reviews']['review_creation_date'] = pd.to_datetime(olist_df['order_reviews']['review_creation_date'])
olist_df['order_reviews']['review_answer_timestamp'] = pd.to_datetime(olist_df['order_reviews']['review_answer_timestamp'])

#MAPEAMENTO SQL
dtype_order_reviews = {
    'review_id': VARCHAR(32),
    'order_id': VARCHAR(32),
    'review_score': Integer(),
    'review_comment_title': VARCHAR(100),
    'review_comment_message': TEXT(),
    'review_creation_date': DateTime(),
    'review_answer_timestamp': DateTime()
}

- 8.3. Criar colunas: Sem necessidade;
- 8.4. Nomes de colunas: Já estão padronizados;
- 8.5. Colunas duplicadas: Não há.

Obs.: Hé valores duplicados na coluna review_id, e como ela será uma Primary Key, é necessário remover os valores duplicados.

In [19]:
#Remove duplicatas baseadas no ID do review
olist_df['order_reviews'] = olist_df['order_reviews'].drop_duplicates(subset='review_id')

9. olist_order_customers_dataset

- 9.1. Tipos de dados: Mapeando as colunas.

In [20]:
#MAPEAMENTO SQL
dtype_customers = {
    'customer_id': VARCHAR(32),
    'customer_unique_id': VARCHAR(32),
    'customer_zip_code_prefix': Integer(),
    'customer_city': VARCHAR(100),
    'customer_state': CHAR(2)
}

- 9.2. Valores nulos: não há;
- 9.3. Criar colunas: Sem necessidade;
- 9.4. Nomes de colunas: Já estão padronizados;
- 9.5. Colunas duplicadas: Não há.

Obs.: customer_city possui inconsistências, portanto é necessário remover caracteres especiais, espaços extras e padronizar tudo em minúsculo para garantir consistência.

In [21]:
olist_df['customers']['customer_city'] = (
    olist_df['customers']['customer_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

10. Exportando os arquivos csv.

In [22]:
for name, df in olist_df.items():
    df.to_csv(f'../data/processed/{name}.csv', index=False)

11. Conectando ao MySQL.

In [23]:
#Configurações da conexão
username = "root"
password = "" 
host = "localhost"
port = "3306"
database = "olist_brazilian_ecommerce" #Criado previamente para ter para onde enviar
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

#Mapeamento dos dicionários criados
map_dtypes = {
    'customers': dtype_customers,
    'order_items': dtype_order_items,
    'order_payments': dtype_order_payments,
    'order_reviews': dtype_order_reviews,
    'orders': dtype_orders,
    'products': dtype_products,
    'sellers': dtype_sellers,
    'category_translation': dtype_category_translation
}

#Envio para o Banco
for name, df in olist_df.items():
    df.to_sql(
        name=name, 
        con=engine, 
        if_exists="replace", 
        index=False, 
        dtype=map_dtypes[name]
    )